# EDA — 탐색적 데이터 분석 (Exploratory Data Analysis)

> **제5회 ETRI 휴먼이해 인공지능 논문경진대회**  
> `00_data_check.ipynb` 실행 완료 후 이 노트북을 실행하세요.

---

## 분석 목차
1. 환경 설정 & 데이터 로드
2. Target 변수 분석
3. 피처 분포 분석
4. 상관관계 분석
5. 시간적 패턴 분석
6. 개인별 패턴 분석
7. 결측치 패턴 분석
8. 이상치 탐지
9. EDA 요약

## 1. 환경 설정 & 데이터 로드

In [ ]:
# -*- coding: utf-8 -*-
import sys, os
from pathlib import Path

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams['figure.figsize'] = (14, 6)
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

FIGURE_DIR = Path(project_root) / 'outputs' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root : {project_root}")
print(f"Figure dir   : {FIGURE_DIR}")

In [ ]:
from src.config import Config
from src.load_data import load_all, print_data_summary

cfg = Config("configs/base.yaml")

try:
    train, test, sub, target_cols, feature_cols = load_all(cfg)
    id_col = cfg.id_col
    num_feat_cols = [c for c in feature_cols if train[c].dtype.kind in ('i', 'f', 'u')]
    cat_feat_cols = [c for c in feature_cols if c not in num_feat_cols]
    print(f"\n데이터 로드 완료: train={train.shape}, test={test.shape}")
    print(f"target={target_cols}, feature={len(feature_cols)}개")
    DATA_LOADED = True
except FileNotFoundError as e:
    print(e)
    print("\n00_data_check.ipynb를 먼저 실행하세요.")
    DATA_LOADED = False

## 2. Target 변수 분석

In [ ]:
if DATA_LOADED:
    display(train[target_cols].describe().round(4))
    n = len(target_cols)
    fig, axes = plt.subplots(2, n, figsize=(5*n, 10))
    if n == 1:
        axes = axes.reshape(2, 1)
    for i, col in enumerate(target_cols):
        data = train[col].dropna()
        axes[0, i].hist(data, bins=30, density=True, alpha=0.7, color='steelblue')
        axes[0, i].set_title(f'{col} — 분포')
        stats.probplot(data, dist="norm", plot=axes[1, i])
        axes[1, i].set_title(f'{col} — Q-Q Plot')
    plt.tight_layout()
    fig.savefig(FIGURE_DIR / 'target_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. 피처 분포 분석

In [ ]:
if DATA_LOADED and num_feat_cols:
    display_cols = num_feat_cols[:12]
    n_rows = (len(display_cols) + 3) // 4
    fig, axes = plt.subplots(n_rows, 4, figsize=(16, 4*n_rows))
    axes = axes.flatten()
    for i, col in enumerate(display_cols):
        axes[i].hist(train[col].dropna(), bins=30, color='steelblue', alpha=0.7)
        axes[i].set_title(col, fontsize=10)
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle(f'수치형 피처 분포 (상위 {len(display_cols)}개)', fontsize=13)
    plt.tight_layout()
    fig.savefig(FIGURE_DIR / 'feature_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. 상관관계 분석

In [ ]:
if DATA_LOADED and num_feat_cols:
    all_num = num_feat_cols + target_cols
    corr = train[[c for c in all_num if c in train.columns]].corr()
    for target in target_cols:
        if target not in corr.columns:
            continue
        top15 = corr[target].drop(target_cols, errors='ignore').abs().sort_values(ascending=False).head(15)
        fig, ax = plt.subplots(figsize=(8, 5))
        top15.sort_values().plot(kind='barh', ax=ax, color='steelblue')
        ax.set_title(f"Top 15 Features Correlated with '{target}'")
        ax.set_xlabel('|Correlation|')
        plt.tight_layout()
        fig.savefig(FIGURE_DIR / f'correlation_{target}.png', dpi=150, bbox_inches='tight')
        plt.show()

## 5. 시간적 패턴 분석

In [ ]:
if DATA_LOADED:
    kws = ["date","time","dt","timestamp","날짜","일자"]
    date_candidates = [c for c in train.columns if any(k in c.lower() for k in kws)]
    if date_candidates:
        date_col = date_candidates[0]
        dt = pd.to_datetime(train[date_col], errors='coerce')
        tmp = train.copy()
        tmp['_dow'] = dt.dt.dayofweek
        tmp['_month'] = dt.dt.month
        tmp['_wknd'] = (dt.dt.dayofweek >= 5).astype(int)
        for target in target_cols[:1]:
            if target not in tmp.columns: continue
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
            tmp.groupby('_dow')[target].mean().plot(kind='bar', ax=axes[0], color='steelblue')
            axes[0].set_xticklabels(day_names, rotation=0)
            axes[0].set_title(f'{target} — 요일별 평균')
            tmp.groupby('_month')[target].mean().plot(kind='bar', ax=axes[1], color='salmon')
            axes[1].set_title(f'{target} — 월별 평균')
            tmp.groupby('_wknd')[target].mean().plot(kind='bar', ax=axes[2], color=['steelblue','salmon'])
            axes[2].set_xticklabels(['평일','주말'], rotation=0)
            axes[2].set_title(f'{target} — 평일/주말')
            plt.tight_layout()
            fig.savefig(FIGURE_DIR / f'temporal_{target}.png', dpi=150, bbox_inches='tight')
            plt.show()
    else:
        print("날짜형 컬럼이 없어 시간 패턴 분석을 건너뜁니다.")

## 6. 개인별 패턴 분석

In [ ]:
if DATA_LOADED:
    ukws = ["user","subject","person","participant","uid","pid"]
    user_candidates = [c for c in train.columns if any(k in c.lower() for k in ukws)]
    if user_candidates:
        group_col = user_candidates[0]
        n_users = train[group_col].nunique()
        print(f"개인 식별 컬럼: '{group_col}' ({n_users}명)")
        for target in target_cols[:1]:
            if target not in train.columns: continue
            ustats = train.groupby(group_col)[target].agg(['mean','std','count'])
            fig, ax = plt.subplots(figsize=(8, 4))
            ustats['mean'].hist(bins=20, ax=ax, color='steelblue')
            ax.set_title(f"{target} — 개인별 평균 분포")
            ax.set_xlabel('개인 평균')
            plt.tight_layout()
            fig.savefig(FIGURE_DIR / f'personal_{target}.png', dpi=150, bbox_inches='tight')
            plt.show()
    else:
        print("개인 식별 컬럼이 없어 개인별 패턴 분석을 건너뜁니다.")

## 7. 결측치 패턴 분석

In [ ]:
if DATA_LOADED:
    missing = train[feature_cols].isna().mean().sort_values(ascending=False)
    nonzero = missing[missing > 0].head(20)
    if len(nonzero) > 0:
        fig, ax = plt.subplots(figsize=(10, max(4, len(nonzero)*0.4)))
        nonzero.sort_values().plot(kind='barh', ax=ax, color='salmon')
        ax.set_title('Train 결측치 비율 (상위 20개)')
        ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5)
        plt.tight_layout()
        fig.savefig(FIGURE_DIR / 'missing_values.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print("결측치 없음")

## 8. 이상치 탐지

In [ ]:
if DATA_LOADED and num_feat_cols:
    outlier_ratios = {}
    for col in num_feat_cols[:20]:
        if col in train.columns:
            q1, q3 = train[col].quantile([0.25, 0.75])
            iqr = q3 - q1
            mask = (train[col] < q1 - 1.5*iqr) | (train[col] > q3 + 1.5*iqr)
            outlier_ratios[col] = mask.mean()
    print("[이상치 비율 상위 10개 피처 (IQR 기준)]")
    display(pd.Series(outlier_ratios).sort_values(ascending=False).head(10).to_frame('이상치 비율'))

## 9. EDA 요약 & 분석 메모

In [ ]:
if DATA_LOADED:
    print("=" * 50)
    print("  EDA 요약")
    print("=" * 50)
    print(f"  Train  : {train.shape}")
    print(f"  Test   : {test.shape}")
    print(f"  Target : {target_cols}")
    print(f"  피처   : 수치형={len(num_feat_cols)}, 범주형={len(cat_feat_cols)}")
    print(f"  결측치 : {train[feature_cols].isna().sum().sum()}개")
    print("\n  저장된 그림:")
    for f in sorted(FIGURE_DIR.glob('*.png')):
        print(f"    {f.name}")

---

## 분석 메모 (실제 데이터 확인 후 작성)

### Target 관찰
- 

### 주요 피처 관찰
- 

### 시간적 패턴
- 

### 개인별 패턴
- 

### 피처 엔지니어링 아이디어
- 

### 다음 실험 계획
- `docs/experiment_log.md` 업데이트 필요